# stockpy Quickstart — Encoder-Decoder Forecasting on AAPL

This notebook walks through the full stockpy `0.4.0` workflow on real stock data:

1. Load the bundled `stock/AAPL.csv` (OHLCV).
2. Chronologically split + scale.
3. Fit an `LSTMForecaster` that maps a `context_len` window of past observations to a `pred_len` window of future values.
4. Forecast the next few days, inverse-scale, and plot against the ground truth.
5. Persist and reload the model with safetensors.

Expected runtime: about a minute on CPU.

> All forecasters in `stockpy.forecasters` follow the same encoder-decoder contract: `predict()` returns an array of shape `(n_windows, pred_len, n_features)`. Swap `LSTMForecaster` for `GRUForecaster`, `BiLSTMForecaster`, `TCNForecaster`, `TransformerForecaster`, or `DMMForecaster` — the rest of the notebook stays the same.

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from stockpy.forecasters import LSTMForecaster
from stockpy.preprocessing import StandardScalerTransform
from stockpy.callbacks import PrintLog

torch.manual_seed(0)
np.random.seed(0)

## 1. Load AAPL

We keep the five OHLCV columns and drop `Adj Close` (redundant with `Close` for our purposes) and the `Date` index (used only for plotting).

In [ ]:
df = (
    pd.read_csv("../stock/AAPL.csv", parse_dates=True, index_col="Date")
    .dropna(how="any")
)
feature_cols = ["Open", "High", "Low", "Close", "Volume"]
data = df[feature_cols].astype(np.float32)
print("shape:", data.shape)
data.head()

## 2. Chronological train/test split + scaling

Time-series splits must be ordered — never shuffle. We fit the scaler on the training window only to avoid leaking statistics from the future. `StandardScalerTransform` mirrors its input library: numpy in → numpy out (and tensor in → tensor out), so we get back numpy arrays without any explicit conversion.

In [ ]:
n_train = int(len(data) * 0.8)
X_train_raw = data.values[:n_train]
X_test_raw = data.values[n_train:]

scaler = StandardScalerTransform()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print(f"train: {X_train.shape}, test: {X_test.shape}")

## 3. Define the forecaster

The model reads `context_len=30` past trading days and predicts the next `pred_len=5`. `TimeSeriesDataset` (used internally) requires `series_len >= context_len + pred_len`, so the choice is bounded by the test-set length.

In [ ]:
CONTEXT_LEN = 30
PRED_LEN = 30

model = LSTMForecaster(
    context_len=CONTEXT_LEN,
    pred_len=PRED_LEN,
    rnn_size=64,
    hidden_size=64,
    num_layers=2,
    dropout=0.1,
)

## 4. Train

`y=X_train` is the auto-regressive setup: targets are the same series, sliced into the future window by `TimeSeriesDataset`. `train_split=None` disables validation splitting since the windowed dataset has fewer rows than the raw series — for production runs supply your own `ValidSplit` or held-out arrays.

In [ ]:
model.fit(
    X_train,
    y=X_train,
    epochs=15,
    batch_size=32,
    lr=1e-3,
    optimizer=torch.optim.Adam,
    train_split=None,
    callbacks=[PrintLog()],
)

## 5. Forecast the next 5 days

Take the last `context_len` rows of the test set as the encoder input, feed them through `predict`, and inverse-scale the result back to dollar units. The ground truth for comparison is the next `pred_len` rows.

In [ ]:
context = X_test[-(CONTEXT_LEN + PRED_LEN) : -PRED_LEN]
actual = X_test_raw[-PRED_LEN:]

# predict() accepts a Dataset; wrap the single context window into one batch.
context_batch = torch.tensor(context).unsqueeze(0)
ds = torch.utils.data.TensorDataset(context_batch, torch.zeros(1, 1))

pred_scaled = model.predict(ds)
pred = scaler.inverse_transform(
    pred_scaled.reshape(-1, len(feature_cols))
).reshape(1, PRED_LEN, len(feature_cols))

close_idx = feature_cols.index("Close")
print("predicted Close:", pred[0, :, close_idx])
print("actual Close:   ", actual[:, close_idx])

## 6. Plot

Show the last 60 days of context plus the forecast vs. the actual future prices.

In [ ]:
history_window = 60
history_close = X_test_raw[-(history_window + PRED_LEN) : -PRED_LEN, close_idx]

x_hist = np.arange(-history_window, 0)
x_future = np.arange(0, PRED_LEN)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(x_hist, history_close, label="context (actual)", color="steelblue")
ax.plot(x_future, actual[:, close_idx], label="actual future", color="black", marker="o")
ax.plot(x_future, pred[0, :, close_idx], label="forecast", color="crimson", marker="x")
ax.axvline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xlabel("days from forecast origin")
ax.set_ylabel("AAPL close (USD)")
ax.set_title(f"LSTMForecaster \u2014 {CONTEXT_LEN}-day context, {PRED_LEN}-day horizon")
ax.legend()
fig.tight_layout()
plt.show()

## 7. Save and reload (safetensors)

stockpy persists model parameters via [safetensors](https://github.com/huggingface/safetensors). To rebuild the model on a fresh process, instantiate it with the same hyperparameters, run a single `fit` call with `epochs=0` to trigger module initialization, then `load_params`.

In [ ]:
ckpt = "aapl_lstm.safetensors"
model.save_params(f_params=ckpt, use_safetensors=True)

reloaded = LSTMForecaster(
    context_len=CONTEXT_LEN,
    pred_len=PRED_LEN,
    rnn_size=64,
    hidden_size=64,
    num_layers=2,
    dropout=0.1,
)
reloaded.fit(X_train, y=X_train, epochs=0, train_split=None, verbose=0)
reloaded.load_params(f_params=ckpt, use_safetensors=True)

preds_reloaded = reloaded.predict(ds)
np.testing.assert_allclose(pred_scaled, preds_reloaded, rtol=1e-5)
print("OK — reload matches original")

## Where to go next

- Try a different forecaster: `from stockpy.forecasters import TransformerForecaster` — same `fit` / `predict` API.
- Use `DMMForecaster` for a probabilistic encoder-decoder (Pyro SVI under the hood).
- Add `Checkpoint` and `LRScheduler` from `stockpy.callbacks` for longer training runs.
- Increase `pred_len` for longer horizons — remember the `series_len >= context_len + pred_len` constraint.